# Purchase prediction: baselines and MLP

**Author:** Davian (dav_1ann) · **Project:** ShopSphere Win-Back Decision Agent · **Checkpoint:** Progress Review 1

**Goal.** Predict whether a customer makes at least one purchase between 1 and 30 October 2024,
using only data dated on or before 30 September 2024. The output is a probability per customer,
which the agent will later use to decide who to contact.

**What this notebook does**

1. Loads the feature table and checks it against the team's data contract
2. Uses the shared train / validation / test split
3. Preprocesses the features (fitted on train only)
4. Trains four non-deep-learning baselines
5. Trains the deep learning model, a multilayer perceptron (MLP)
6. Tunes the MLP and logs every run
7. Checks the result can be trusted (overfitting, seeds, leakage)
8. Evaluates every model once on the test split
9. Checks performance by country and tier
10. Saves the chosen model

**How the three splits are used.** Train: the model learns. Validation: early stopping, tuning
and choosing the threshold. Test: touched once, in section 8, for the reported numbers.

The reusable code is in `src/shopsphere/`. This notebook calls it and explains each step.

In [ ]:
import json

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from sklearn.metrics import ConfusionMatrixDisplay, precision_recall_curve

from shopsphere import config
from shopsphere.baselines import (
    always_zero_scores,
    fit_lightgbm,
    fit_logistic_regression,
    recency_scores,
)
from shopsphere.data import attach_labels, input_columns, load_features
from shopsphere.metrics import best_f1_threshold, evaluate, evaluate_by_group
from shopsphere.model import MLPParams
from shopsphere.placeholder import make_placeholder
from shopsphere.preprocess import build_preprocessor
from shopsphere.train import DEVICE, predict_proba, train_mlp
from shopsphere.tuning import params_from_row, random_search

pd.set_option("display.precision", 4)
print("Training on:", DEVICE)

## 1. Load the data

`load_features` reads `data/processed/features.csv` and stops with a clear error if the file breaks
the contract: one row per customer, a `customer_id` column, no `target` column and no raw dates.

**Until data preparation is finished, this cell falls back to synthetic placeholder data** so the
rest of the notebook can be run and checked. Placeholder numbers mean nothing and must not be
reported. Once `features.csv` and `train.csv` are in place, re-run the notebook: nothing else needs
to change, apart from the column names in `config.py` if they differ.

In [ ]:
USING_PLACEHOLDER = not (config.FEATURES_FILE.exists() and config.LABELS_FILE.exists())

if USING_PLACEHOLDER:
    features, labels = make_placeholder()
    print("*** SYNTHETIC PLACEHOLDER DATA: results below are NOT real and must not be reported ***")
else:
    features = load_features(config.FEATURES_FILE)
    labels = pd.read_csv(config.LABELS_FILE)

print(f"Feature table: {features.shape[0]:,} customers x {features.shape[1]} columns")
print(f"Labelled customers: {len(labels):,}")
features.head()

## 2. Labels and the shared split

`attach_labels` joins the target onto the features and adds the team's split: 70% train, 15%
validation, 15% test, stratified on the target with a fixed seed. Every member uses the same
function, so our results are comparable.

Stratifying keeps the share of buyers the same in each part. With a rare class, a plain random
split could give one part noticeably more or fewer buyers by chance.

In [ ]:
data = attach_labels(features, labels)

train = data[data["split"] == "train"]
val = data[data["split"] == "val"]
test = data[data["split"] == "test"]

y_train, y_val, y_test = (part[config.TARGET_COL].to_numpy() for part in (train, val, test))

data.groupby("split")[config.TARGET_COL].agg(customers="size", buy_rate="mean").loc[["train", "val", "test"]]

**The class imbalance decides how we evaluate.** The buy rate above is the *base rate*. A model that
predicts "won't buy" for everyone scores an accuracy of 1 minus the base rate while finding no
buyers at all. So accuracy is reported only next to that floor, and the main metric is **PR-AUC**:
it measures how well the model ranks the rare buyers first, and a random model scores the base rate.

> **Write up (real data):** state the base rate and the accuracy floor it implies.

## 3. Preprocessing

Model inputs are every column except the id, the target, the split label and the `noise_*` system
fields. They are divided by data type:

- **Numeric:** fill gaps with the train median, add a flag recording that the value was missing,
  then standardise to mean 0 and standard deviation 1. A neural network trains poorly when inputs
  are on very different scales.
- **Categorical:** fill gaps with the label `missing`, then one-hot encode.

The preprocessor is **fitted on the train split only** and then applied to validation and test.
Fitting on all rows would leak validation and test statistics into training.

In [ ]:
numeric_cols, categorical_cols = input_columns(data)
print(f"{len(numeric_cols)} numeric:", numeric_cols)
print(f"{len(categorical_cols)} categorical:", categorical_cols)

preprocessor = build_preprocessor(numeric_cols, categorical_cols)
X_train = preprocessor.fit_transform(train)   # fit on train only
X_val = preprocessor.transform(val)           # apply, never fit
X_test = preprocessor.transform(test)

print("Model input matrix:", X_train.shape)

> **Check (real data):** read the two column lists. Is anything in them that should not be a model
> input, such as a column computed after the cutoff date? If so, add it to `DROP_COLS` in `config.py`.

## 4. Baselines

The brief requires a fair comparison against at least one non-deep-learning model. We use four,
each answering a different question:

| Baseline | Question it answers |
|---|---|
| Always predict 0 | What is the floor? |
| Recency rule | Can we beat what the CRM team does today (rank by days since last order)? |
| Logistic regression | Does a simple linear model already solve it? |
| LightGBM | Can the MLP match a strong tree model, usually the best choice for tables? |

All of them use the same split and the same preprocessed inputs as the MLP. The recency rule needs
no fitting: it only sorts customers.

`val_scores` and `test_scores` hold each model's scores, where higher means "more likely to buy".
Test scores are computed here but **not looked at until section 8**.

In [ ]:
val_scores, test_scores = {}, {}

val_scores["Always 0"] = always_zero_scores(len(val))
test_scores["Always 0"] = always_zero_scores(len(test))

val_scores["Recency rule"] = recency_scores(val[config.RECENCY_COL])
test_scores["Recency rule"] = recency_scores(test[config.RECENCY_COL])

logistic = fit_logistic_regression(X_train, y_train)
val_scores["Logistic regression"] = logistic.predict_proba(X_val)[:, 1]
test_scores["Logistic regression"] = logistic.predict_proba(X_test)[:, 1]

lightgbm = fit_lightgbm(X_train, y_train)
val_scores["LightGBM"] = lightgbm.predict_proba(X_val)[:, 1]
test_scores["LightGBM"] = lightgbm.predict_proba(X_test)[:, 1]

## 5. The deep learning model

**Architecture.** A multilayer perceptron: `inputs -> Linear(128) -> ReLU -> Dropout(0.3) ->
Linear(64) -> ReLU -> Dropout(0.3) -> Linear(1)`. The final unit outputs a logit, and a sigmoid
turns it into a probability.

**Training choices, and why**

| Choice | Reason |
|---|---|
| Weighted binary cross-entropy (`pos_weight` = non-buyers / buyers) | Buyers are rare. Without the weight, the loss is dominated by non-buyers and the model learns to say "won't buy" |
| Adam optimiser | Adapts the step size per weight; a reliable default for an MLP |
| Dropout | Switches off random units during training, which reduces overfitting |
| Early stopping on validation PR-AUC | Stops when the model no longer improves on unseen data, and keeps the best epoch's weights |
| Fixed seed | The same run gives the same result, so differences between runs come from the settings |

We start from the configuration in the project plan, then tune it in section 6.

In [ ]:
default_params = MLPParams()
default_model, default_history = train_mlp(X_train, y_train, X_val, y_val, default_params)

best_epoch = default_history.loc[default_history["val_pr_auc"].idxmax()]
print(f"Stopped after {len(default_history)} epochs; best was epoch {int(best_epoch['epoch'])}")
print(f"Validation PR-AUC: {best_epoch['val_pr_auc']:.4f}   (random = {y_val.mean():.4f})")

## 6. Hyperparameter tuning

A random search over layer sizes, dropout, learning rate, weight decay and batch size. Random search
covers a wide space with few runs and is simple to explain. Trial 0 is always the default
configuration, so every other trial reads as better or worse than where we started.

Models are compared on **validation** PR-AUC only. Every run is written to the tuning log in
`reports/`, which is our evidence of optimisation.

In [ ]:
N_TRIALS = 15  # raise for a wider search; each trial trains one model

log_name = "tuning_log_PLACEHOLDER.csv" if USING_PLACEHOLDER else "tuning_log_davian.csv"
tuning_log = random_search(
    X_train, y_train, X_val, y_val, n_trials=N_TRIALS, log_path=config.ROOT / "reports" / log_name
)
tuning_log

In [ ]:
best_params = params_from_row(tuning_log.iloc[0])
default_score = tuning_log.loc[tuning_log["trial"] == 0, "val_pr_auc"].item()

print("Best configuration:", best_params)
print(f"Validation PR-AUC: default {default_score:.4f} -> tuned {tuning_log.iloc[0]['val_pr_auc']:.4f}")

# Retrain the chosen configuration to get the model object and its learning curves.
mlp, history = train_mlp(X_train, y_train, X_val, y_val, best_params)
val_scores["MLP (tuned)"] = predict_proba(mlp, X_val)
test_scores["MLP (tuned)"] = predict_proba(mlp, X_test)

> **Write up (real data):** which settings mattered most? Look at how `val_pr_auc` changes with
> each column of the log. How large is the gain over the default, and is it bigger than the
> seed-to-seed variation measured in section 7.2? If not, the "improvement" may be noise.

## 7. Can we trust the result?

### 7.1 Overfitting: learning curves

If training loss keeps falling while validation loss rises, the model is memorising the training
data. Early stopping should end training near the point where the validation curve turns. The
dashed line marks the epoch whose weights were kept.

In [ ]:
kept_epoch = history.loc[history["val_pr_auc"].idxmax(), "epoch"]

fig, (ax_loss, ax_pr) = plt.subplots(1, 2, figsize=(11, 3.8))
for ax, metric, title in [(ax_loss, "loss", "Loss (weighted BCE)"), (ax_pr, "pr_auc", "PR-AUC")]:
    ax.plot(history["epoch"], history[f"train_{metric}"], label="train")
    ax.plot(history["epoch"], history[f"val_{metric}"], label="validation")
    ax.axvline(kept_epoch, linestyle="--", color="grey", linewidth=1)
    ax.set(title=title, xlabel="epoch")
    ax.legend()
fig.tight_layout()
plt.show()

> **Write up (real data):** describe the gap between the train and validation curves and whether
> it grows.

### 7.2 Stability across seeds

A neural network starts from random weights, so one run could be lucky. We retrain the chosen
configuration with several seeds. A small standard deviation means the score is repeatable.

In [ ]:
seed_scores = []
for seed in [1, 2, 3, 4, 5]:
    _, seed_history = train_mlp(X_train, y_train, X_val, y_val, best_params, seed=seed)
    seed_scores.append(seed_history["val_pr_auc"].max())

print("Validation PR-AUC per seed:", np.round(seed_scores, 4))
print(f"Mean {np.mean(seed_scores):.4f}, standard deviation {np.std(seed_scores):.4f}")

### 7.3 Leakage: shuffled-label test

We shuffle the training labels, which destroys any real link between features and target, and
train again. The model should now score about the base rate on validation. If it still scores
well, information about the target is reaching the model some other way, i.e. the pipeline leaks.

In [ ]:
shuffled = np.random.default_rng(config.SEED).permutation(y_train)
_, shuffled_history = train_mlp(X_train, shuffled, X_val, y_val, best_params)

print(f"Validation PR-AUC with shuffled labels: {shuffled_history['val_pr_auc'].max():.4f}")
print(f"Base rate (what random scores):         {y_val.mean():.4f}")

> **Write up (real data):** is the shuffled-label score close to the base rate? Also check the
> opposite warning sign: a real ROC-AUC above about 0.95 is suspicious and needs an explanation,
> usually one feature that contains post-cutoff information.

## 8. Final evaluation on the test split

This is the only place the test split is used. Each model's yes/no threshold is the one that
maximised F1 on **validation**; choosing it on test would make the results look better than they are.

How to read the table:

- **pr_auc** (main metric): ranking quality for a rare class. Random scores the base rate.
- **roc_auc:** probability that a random buyer is ranked above a random non-buyer. Random scores 0.5.
- **precision:** of the customers we flag, the share who buy. **recall:** of all buyers, the share we flag.
- **f1:** balance of precision and recall at the chosen threshold.
- **precision_top10 / lift_top10:** buy rate in the top 10% of the list, and how many times better
  that is than picking at random. This is the business view, because the budget only covers the top.
- **accuracy:** only meaningful next to the "Always 0" row.

In [ ]:
thresholds = {
    name: 0.5 if name == "Always 0" else best_f1_threshold(y_val, scores)
    for name, scores in val_scores.items()
}

comparison = pd.DataFrame(
    {name: evaluate(y_test, test_scores[name], thresholds[name]) for name in test_scores}
).T.sort_values("pr_auc", ascending=False)

print(f"Test base rate: {y_test.mean():.4f}")
comparison

In [ ]:
fig, (ax_pr, ax_cm) = plt.subplots(1, 2, figsize=(12, 4.2))

for name, scores in test_scores.items():
    if name == "Always 0":
        continue
    precision, recall, _ = precision_recall_curve(y_test, scores)
    ax_pr.plot(recall, precision, label=name)
ax_pr.axhline(y_test.mean(), linestyle="--", color="grey", label="random (base rate)")
ax_pr.set(xlabel="recall", ylabel="precision", title="Precision-recall curves, test split")
ax_pr.legend()

mlp_pred = (test_scores["MLP (tuned)"] >= thresholds["MLP (tuned)"]).astype(int)
ConfusionMatrixDisplay.from_predictions(
    y_test, mlp_pred, display_labels=["no purchase", "purchase"], ax=ax_cm, colorbar=False
)
ax_cm.set_title("MLP confusion matrix, test split")
fig.tight_layout()
plt.show()

> **Write up (real data):**
> - Does the MLP beat every baseline on PR-AUC? If LightGBM wins, say so and explain why (tree
>   models are often stronger on tabular data). An honest comparison is what is assessed.
> - Translate the top-10% result into business terms: "of the top N customers we flag, X buy,
>   against Y if picked at random".
> - From the confusion matrix: which error is more common, and which costs ShopSphere more, a
>   missed buyer or a wasted contact?

## 9. Performance by country and tier

A model can look good overall while doing poorly for one group. We repeat the test metrics for each
country and each loyalty tier, using the same threshold. Large gaps need an explanation or a
mitigation, and feed the Ethical AI checklist.

In [ ]:
for column in config.GROUP_COLS:
    if column not in test.columns:
        print(f"'{column}' is not in the feature table: skipped")
        continue
    print(f"MLP on the test split, by {column}")
    display(
        evaluate_by_group(y_test, test_scores["MLP (tuned)"], test[column], thresholds["MLP (tuned)"])
    )

> **Write up (real data):** which group has the lowest recall or PR-AUC? Is it a small group (so
> the number is noisy) or a real gap?

## 10. Save the model

Three files are needed to reuse the model: the fitted preprocessor, the network weights, and the
settings required to rebuild the network and apply the threshold. The agent will load these
through a `predict()` function at Progress Review 2. Nothing is saved when running on placeholder data.

In [ ]:
if USING_PLACEHOLDER:
    print("Placeholder data: model not saved.")
else:
    config.MODELS_DIR.mkdir(exist_ok=True)
    joblib.dump(preprocessor, config.MODELS_DIR / "mlp_davian_preprocessor.joblib")
    torch.save(mlp.state_dict(), config.MODELS_DIR / "mlp_davian_weights.pt")
    metadata = {
        "params": best_params.to_dict(),
        "n_inputs": int(X_train.shape[1]),
        "threshold": thresholds["MLP (tuned)"],
        "numeric_cols": numeric_cols,
        "categorical_cols": categorical_cols,
        "seed": config.SEED,
        "test_metrics": comparison.loc["MLP (tuned)"].to_dict(),
    }
    (config.MODELS_DIR / "mlp_davian_metadata.json").write_text(json.dumps(metadata, indent=2))
    print("Saved to", config.MODELS_DIR)

## Summary and next steps

> **Write up (real data):** three or four sentences. Which model is chosen and why, its test
> PR-AUC against the best baseline, what the trust checks showed, and the main limitation.

Known limitations to state:

- The model predicts *buying*, not *responding to a voucher*.
- It is trained on October behaviour, while the retention budget also covers the November and
  December peak.
- The probabilities are not calibrated yet: class weighting inflates them. Calibration is planned
  before Progress Review 2, because the agent's offer bands depend on it.

Next: calibration and probability bands, per-customer explanations (SHAP), and the `predict()`
function the agent calls.